In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("D05_IMDB Dataset.csv")
df.head(4)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Convert to Lower Case

In [7]:
df["review"] = df["review"].str.lower()

# Removing URL

In [8]:
import re

In [9]:
def remove_url(text):
    text = re.sub(r"http\S+", "", text)
    return text
df["review"] = df["review"].apply(remove_url)

# Remove Puntuation

In [10]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
df["review"] = df["review"].apply(remove_punctuations)

# Remove HTML Tags

In [11]:
def remove_htmltag(text):
    text = re.sub(r"<.*?>", "", text)
    return text
df["review"] = df["review"].apply(remove_htmltag)

# Stopword Removal

In [12]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stopword = stopwords.words("english")
    for token in tokens:
        if token in stopword:
            text = text.replace(token, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [14]:
df["review"]

0        e    revewers  nted   wtchg  1 oz epode ll  ho...
1         wderful ltle producti br br  filming techniqu...
2         thought ths   wderful wy  spend tme   o hot s...
3        bsclly res  fmly   lttle boy jke thks res  zom...
4        petter mtte love   time  mey   vully stunng fi...
                               ...                        
49995     thought ths move dd   rght good job t wsnt s ...
49996    bd plot bd dlogue bd ctng dotc drectng  nnoyng...
49997       ctholc tught n prochl elentry schools  nuns...
49998    im gog    dgree   previous comnt  side  mlt   ...
49999     one expects  str trek moves   hgh rt   fns  e...
Name: review, Length: 49582, dtype: str

# Stemming

In [15]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [16]:
df["review"]

0        e revew nted wtchg 1 oz epod ll hook y rght ex...
1        wder ltle producti br br film techniqu unssum ...
2        thought th wder wy spend tme o hot summer week...
3        bsclli re fmli lttle boy jke thk re zomb close...
4        petter mtte love time mey vulli stunng film wt...
                               ...                        
49995    thought th move dd rght good job t wsnt s cret...
49996    bd plot bd dlogu bd ctng dotc drectng nnoyng p...
49997    ctholc tught n prochl elentri school nun tught...
49998    im gog dgree previou comnt side mlt e secd rte...
49999    one expect str trek move hgh rt fn expect move...
Name: review, Length: 49582, dtype: str

# Encoding y

In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

# Vectorize X

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df["review"])

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [21]:
import torch
import torch.nn as nn

In [22]:
X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32)

In [23]:
from torch.utils.data import TensorDataset, DataLoader
train_tensor = TensorDataset(X_train_tensor, y_train_tensor)
test_tensor = TensorDataset(X_test_tensor, y_test_tensor)

In [24]:
train_dataloader = DataLoader(train_tensor, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_tensor, batch_size=64)

In [26]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])
        return out

In [28]:
import torch.optim as optim

input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [30]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_dataloader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1) # add singleton direction
        outputs = model(Xb) # (batch_size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.24122503399848938
epoch = 2/10 and loss = 0.30527690052986145
epoch = 3/10 and loss = 0.13766716420650482
epoch = 4/10 and loss = 0.24654319882392883
epoch = 5/10 and loss = 0.2204195111989975
epoch = 6/10 and loss = 0.12343145906925201
epoch = 7/10 and loss = 0.12758706510066986
epoch = 8/10 and loss = 0.1581992655992508
epoch = 9/10 and loss = 0.18238650262355804
epoch = 10/10 and loss = 0.2426706850528717


In [32]:
# evaluate
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    for Xb, yb in test_dataloader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.74165574266411
